# Fitting Pks Machinery

## Setup (from demo)

In [44]:
import numpy as np
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid
import matplotlib.pyplot as plt
import pyccl as ccl
import os

In [45]:
class Pk(object):
    # Class to store power spectrum data
    def __init__(self, kmin, kmax, kavg, pk, nk):
        self.kmin = kmin
        self.kmax = kmax
        self.kavg = kavg
        self.pk = pk
        self.nk = nk

In [46]:
class SnapPk(object):
    Lbox = 1000.0  # Mpc Flamingo box size
    # Class to organise the power spectrum measurements in a given snapshot
    def __init__(self, namesim, snapnum, kmin=1e-2, kmax=1.0):
        self.sn = snapnum
        self.z = np.loadtxt("FLAMINGO/zs_list.txt")[snapnum]
        self.snapstr = '%04d' % self.sn
        self.sim = namesim
        self.kmin = kmin
        self.kmax = kmax
        self.predir = f'FLAMINGO/L1_m9/{self.sim}/power_spectra'

        # Read and rebin all power spectra
        self.pks = {f'{n1}_{n2}': self.read_pk(self.get_fname_pk(n1, n2))
                    for n1, n2 in self.iter_pairs()}

        # And estimate their Gaussian uncertainties
        self.epks = {f'{n1}_{n2}': self.get_pk_errors(n1, n2)
                     for n1, n2 in self.iter_pairs()}

    def get_pk_covar(self, na, nb, nc, nd):
        # Gaussian Pk covariance
        pkac = self.pks[f'{na}_{nc}'].pk
        pkad = self.pks[f'{na}_{nd}'].pk
        pkbc = self.pks[f'{nb}_{nc}'].pk
        pkbd = self.pks[f'{nb}_{nd}'].pk
        nk = self.pks[f'{na}_{nb}'].nk
        return (pkac*pkbd+pkad*pkbc)/nk

    def get_pk_errors(self, n1, n2):
        # Gaussian error bar
        pk11 = self.pks[f'{n1}_{n1}'].pk
        pk12 = self.pks[f'{n1}_{n2}'].pk
        pk22 = self.pks[f'{n2}_{n2}'].pk
        nk = self.pks[f'{n1}_{n2}'].nk
        return np.sqrt((pk11*pk22+pk12**2)/nk)

    def iter_pairs(self):
        # Returns all pairs of fields (including auto-correlations)
        for n1 in ['matter', 'pressure']:
            for n2 in ['matter', 'pressure']:
                yield n1, n2

    def iter_pairs_unique(self):
        # Returns unique pairs of fields (including auto-correlations)
        names = ['matter', 'pressure']
        for i, n1 in enumerate(names):
            for n2 in names[i:]:
                yield n1, n2

    def get_fname_pk(self, n1, n2):
        # Returns the filename of the power spectrum for a given pair of fields
        if n1 == n2:
            fname1 = fname2 = f'{self.predir}/power_{n1}_{self.snapstr}.txt'
        else:
            fname1 = f'{self.predir}/power_{n1}-{n2}_{self.snapstr}.txt'
            fname2 = f'{self.predir}/power_{n2}-{n1}_{self.snapstr}.txt'

        for fname in [fname1, fname2]:
            if os.path.isfile(fname):
                return fname
        raise KeyError(f'Unknown power spectrum {n1}-{n2}, {fname1}, {fname2}')

    def read_pk(self, fname_pk):
        # Reads a P(k) file in Sara's format and rebins it
        # to a common binning scheme
        d = np.loadtxt(fname_pk, unpack=True)
        ks = d[1]
        pk = d[2]
        # Impose scale cuts
        goodk = (ks >= self.kmin) & (ks <= self.kmax)
        ks = ks[goodk]
        pk = pk[goodk]
        dlk = np.diff(np.log(ks), append=np.log(ks[-1]**2/ks[-2]))
        kmin = np.exp(np.log(ks)-dlk/2)
        kmax = np.exp(np.log(ks)+dlk/2)
        kavg = ks
        nk = (self.Lbox*ks)**3*dlk/(2*np.pi**2)
        return Pk(kmin, kmax, kavg, pk, nk)

In [ ]:
# Create a cosmology object with the same parameters as Flamingo
cosmo = ccl.Cosmology(h=0.681, Omega_b=0.0486, Omega_c=0.306-0.0486, n_s=0.967, sigma8=0.807)

# Initialise the Eulerian PT calculator.
# Check out the documentation in https://ccl.readthedocs.io/en/latest/api/pyccl.nl_pt.ept.html
# to see the meaning of all parameters.
# We may want to play around with e.g. b1_pk_kind and sub_lowk, for example.
ept = ccl.nl_pt.EulerianPTCalculator(with_NC=True, sub_lowk=False, b1_pk_kind='pt', bk2_pk_kind='linear')
ept.update_ingredients(cosmo)

# Generate power spectrum interpolators for the different
# EFT templates.
pairs = ['b1:b1', 'b1:b2', 'b1:bs', 'b1:bk2', 'b2:b2', 'b2:bs', 'bs:bs']
pk2d_templates = {pair: ept.get_pk2d_template(pair) for pair in pairs}

## Fitting Machinery (from demo)

In [50]:
from scipy.optimize import curve_fit, least_squares

def plot_fit(k, pkd, epk, biases, templates, nl = False):
    pkfit = np.dot(templates, biases)
    fig, axes = plt.subplot_mosaic(
        '''
        AAA
        AAA
        AAA
        BBB''', figsize=(6, 6), sharex=True)
    ax = axes['A']
    ax.errorbar(k, pkd, yerr=epk, label='data', fmt='o', markersize=3)
    ax.plot(k, pkfit, label='fit', color='red')
    
    if nl:
        for b, t, name in zip(biases, templates.T, ['b1:b1', 'b1:b2', 'b1:bs', 'b1:bk2', 'b2:b2', 'b2:bs', 'bs:bs','N']):
            if np.mean(b*t) < 0:
                tplot, ls = -t, '--'
            else:
                tplot, ls = t, '-'
            ax.plot(k, b*tplot, ls, label=name)
    else:    
        for b, t, name in zip(biases, templates.T, ['b1:b1', 'b1:b2', 'b1:bs', 'b1:bk2', 'N']):
                if np.mean(b*t) < 0:
                    tplot, ls = -t, '--'
                else:
                    tplot, ls = t, '-'
                ax.plot(k, b*tplot, ls, label=name)
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_ylabel('P(k) [(Mpc/h)^3]')
    ax.legend()
    ax = axes['B']
    ax.errorbar(k, (pkd-pkfit)/epk, yerr=1, fmt='o', markersize=3)
    ax.axhline(0, color='red', ls='--')
    ax.set_xscale('log')
    ax.set_ylabel(r'$\Delta P(k)/\sigma_P$')
    ax.set_xlabel('k [h/Mpc]')


def fit_pk(snpk, pk_data, pk_error, kmax=0.25, kmin=0.01, verbose=True, plot=True, nl = False):
    # Fit a power spectrum with the EFT templates using curve_fit
    a = 1/(1+snpk.z)

    # Fit a power spectrum with the EFT templates
    k = pk_data.kavg
    pk = pk_data.pk
    epk = pk_error  # Gaussian error bars

    # Select the data to fit
    goodk = (k >= kmin) & (k <= kmax)
    kfit = k[goodk]
    pkfit = pk[goodk]
    epkfit = epk[goodk]

    tk_b1_b1 = pk2d_templates['b1:b1'](kfit, a)
    tk_b1_b2 = pk2d_templates['b1:b2'](kfit, a)
    tk_b1_bs = pk2d_templates['b1:bs'](kfit, a)
    tk_b1_bk2 = pk2d_templates['b1:bk2'](kfit, a)
    tk_b2_b2 = pk2d_templates['b2:b2'](kfit, a)
    tk_b2_bs = pk2d_templates['b2:bs'](kfit, a)
    tk_bs_bs = pk2d_templates['bs:bs'](kfit, a)

    # Define the model function to fit
    def model(k, b1, b2, bs, bk2, N):
        return (b1 * tk_b1_b1 +
                0.5*b2 * tk_b1_b2 +
                0.5*bs * tk_b1_bs +
                0.5*bk2 * tk_b1_bk2 + N)

    def nl_model(k, b1, b2, bs, bk2, N):
        return (b1**2 * tk_b1_b1 +
                b2*b1 * tk_b1_b2 +
                b1*bs * tk_b1_bs +
                0.25 * b2**2 * tk_b2_b2 + 
                0.5 * b2*bs * tk_b2_bs +
                0.25 * bs**2 * tk_bs_bs +
                b1*bk2 * tk_b1_bk2 +
                N)

    # Fit the model to the data using least squares
    if nl:
        popt, pcov = curve_fit(nl_model, kfit, pkfit, sigma=epkfit, maxfev=1000000) #Struggles to converge at default (maxfev = 1200)
        chi2 = np.sum(((pkfit - nl_model(kfit, *popt)) / epkfit) ** 2)
        ndof = len(pkfit) - len(popt)
        if verbose:
            print(f'Fit results: b1={popt[0]:.3e}, b2={popt[1]:.3e}, bs={popt[2]:.3e}, bk2={popt[3]:.3e}, N={popt[4]:.3e}, chi2/ndof={chi2/ndof:.2f}')
            print(popt)
        if plot:
            b1, b2, bs, bk2, N = popt
            amps = np.array([
                b1**2,          
                2*b1*b2,          
                2*b1*bs,          
                b2**2,    
                b2*bs,     
                bs**2,     
                2*b1*bk2,         
                N,              
            ])
            Tmat = np.array([tk_b1_b1, 0.5*tk_b1_b2, 0.5*tk_b1_bs,
                             0.25*tk_b2_b2, 0.5*tk_b2_bs, 0.25*tk_bs_bs,
                             0.5*tk_b1_bk2, np.ones_like(tk_b1_b1)]).T
            plot_fit(kfit, pkfit, epkfit, amps, Tmat, nl = True)
        return popt, pcov, chi2/ndof
    else:
        popt, pcov = curve_fit(model, kfit, pkfit, sigma=epkfit)
        chi2 = np.sum(((pkfit - model(kfit, *popt)) / epkfit) ** 2)
        ndof = len(pkfit) - len(popt)
        if verbose:
            print(f'Fit results: b1={popt[0]:.3e}, b2={popt[1]:.3e}, bs={popt[2]:.3e}, bk2={popt[3]:.3e}, N={popt[4]:.3e}, chi2/ndof={chi2/ndof:.2f}')
        if plot:
            Tmat = np.array([tk_b1_b1, 0.5*tk_b1_b2, 0.5*tk_b1_bs,
                             0.5*tk_b1_bk2, np.ones_like(tk_b1_b1)]).T
            plot_fit(kfit, pkfit, epkfit, popt, Tmat)
        return popt, pcov, chi2/ndof
    

def fit_pk_analytic(snpk, pk_data, pk_error, kmax=0.25, kmin=0.01, verbose=True, plot=True):
    # Fit a power spectrum with the EFT templates using the analytical least-squares solution
    a = 1/(1+snpk.z)

    # Fit a power spectrum with the EFT templates
    k = pk_data.kavg
    pk = pk_data.pk
    epk = pk_error  # Gaussian error bars

    # Select the data to fit
    goodk = (k >= kmin) & (k <= kmax)
    kfit = k[goodk]
    pkfit = pk[goodk]
    epkfit = epk[goodk]

    tk_b1_b1 = pk2d_templates['b1:b1'](kfit, a)
    tk_b1_b2 = pk2d_templates['b1:b2'](kfit, a)
    tk_b1_bs = pk2d_templates['b1:bs'](kfit, a)
    tk_b1_bk2 = pk2d_templates['b1:bk2'](kfit, a)


    
    tk_N = np.ones_like(tk_b1_b1)
    Tmat = np.array([tk_b1_b1, 0.5*tk_b1_b2, 0.5*tk_b1_bs, 0.5*tk_b1_bk2, tk_N]).T
    Cov = np.diag(epkfit**2)
    iCov = np.linalg.inv(Cov)
    TiCT = np.dot(Tmat.T, np.dot(iCov, Tmat))
    TiCp = np.dot(Tmat.T, np.dot(iCov, pkfit))
    pcov = np.linalg.inv(TiCT)
    popt = np.dot(pcov, TiCp)
    chi2 = np.sum(((pkfit - np.dot(Tmat, popt)) / epkfit) ** 2)
    ndof = len(pkfit) - len(popt)
    if verbose:
        print(f'Fit results: b1={popt[0]:.3e}, b2={popt[1]:.3e}, bs={popt[2]:.3e}, bk2={popt[3]:.3}, N={popt[4]:.3e}, chi2/ndof={chi2/ndof:.2f}')
    if plot:
        plot_fit(kfit, pkfit, epkfit, popt, Tmat)
    return popt, pcov, chi2/ndof

In [51]:
def fit_joint(snpk, kmax=0.25, kmin=0.01, verbose=True):
    
    a = 1/(1+snpk.z)
    k = snpk.pks['matter_pressure'].kavg
    goodk = (k >= kmin) & (k <= kmax)
    kfit = k[goodk]

    pk_mp  = snpk.pks['matter_pressure'].pk[goodk]
    epk_mp = snpk.epks['matter_pressure'][goodk]
    pk_pp  = snpk.pks['pressure_pressure'].pk[goodk]
    epk_pp = snpk.epks['pressure_pressure'][goodk]

    # evaluate the templates once, at kfit and a
    tk_b1_b1  = pk2d_templates['b1:b1'](kfit, a)
    tk_b1_b2  = pk2d_templates['b1:b2'](kfit, a)
    tk_b1_bs  = pk2d_templates['b1:bs'](kfit, a)
    tk_b1_bk2 = pk2d_templates['b1:bk2'](kfit, a)
    tk_b2_b2  = pk2d_templates['b2:b2'](kfit, a)
    tk_b2_bs  = pk2d_templates['b2:bs'](kfit, a)
    tk_bs_bs  = pk2d_templates['bs:bs'](kfit, a)

    def joint_residual(params):
        b1, b2, bs, bk2, N_mp, N_pp = params

        mp = (b1 * tk_b1_b1 +
              0.5*b2 * tk_b1_b2 +
              0.5*bs * tk_b1_bs +
              0.5*bk2 * tk_b1_bk2 +
              N_mp)

        pp = (b1**2 * tk_b1_b1 +
              b1*b2 * tk_b1_b2 +
              b1*bs * tk_b1_bs +
              0.25*b2**2 * tk_b2_b2 +
              0.5*b2*bs * tk_b2_bs +
              0.25*bs**2 * tk_bs_bs +
              b1*bk2 * tk_b1_bk2 +
              N_pp)

        res_mp = (pk_mp - mp) / epk_mp
        res_pp = (pk_pp - pp) / epk_pp
        return np.concatenate([res_mp, res_pp])

    p0 = [1e-5, 0, 0, 0, 0, 0]
    result = least_squares(joint_residual, p0, method='trf')

    popt = result.x
    chi2 = np.sum(result.fun**2)                    
    ndof = len(result.fun) - len(popt)             
    cov  = np.linalg.inv(result.jac.T @ result.jac)
    perr = np.sqrt(np.diag(cov))

    if verbose:
        names = ['b1', 'b2', 'bs', 'bk2', 'N_mp', 'N_pp']
        for n, v, e in zip(names, popt, perr):
            print(f'{n} = {v:.3e} ± {e:.3e}')
        print(f'chi2/ndof = {chi2/ndof:.2f}')

    return popt, perr, chi2/ndof

## Plotting Machinery

In [53]:
def coefficient_plotter(snpk, kmaxes, which = None): #Plots specified coefficients for a given snapshot.
    
    names = ['b1', 'b2', 'bs', 'bk2', 'N']
    
    if which is None:
        which = names
        
    for idx, name in enumerate(names):
        
        if name in which:
            parameters, errors = [], []
            parameters_nl, errors_nl = [], []
            
            for km in kmaxes:
                
                popt, pcov, _ = fit_pk(snpk, snpk.pks['matter_pressure'], snpk.epks['matter_pressure'], kmax=km, verbose=False, plot=False)
                popt_nl,  pcov_nl, _ = fit_pk(snpk, snpk.pks['pressure_pressure'], snpk.epks['pressure_pressure'], kmax=km, verbose=False, plot=False, nl = True)
                parameters.append(popt[idx])
                errors.append(np.sqrt(np.diag(pcov))[idx])
                parameters_nl.append(popt_nl[idx])
                errors_nl.append(np.sqrt(np.diag(pcov_nl))[idx])
                
            fig, ax = plt.subplots()
            ax.errorbar(kmaxes, parameters, yerr = errors, label = 'matter-pressure',
                        fmt='o-',        
                        capsize=3,     
                        capthick=1,     
                        elinewidth=1,   
                        markersize=4,
                       alpha = 0.7)
            ax.errorbar(kmaxes, parameters_nl, yerr = errors_nl, label = 'pressure-pressure',
                        fmt='o-',        
                        capsize=3,     
                        capthick=1,     
                        elinewidth=1,   
                        markersize=4,
                        alpha = 0.7)
            
            ax.set_xlabel(r'$k_{\max}$ [h/Mpc]')
            ax.set_ylabel('Bias value')
            ax.legend()
            ax.set_title(f'{name} against k');

In [52]:
def joint_coefficient_plotter(snpk, kmaxes, which=None):
    joint_names = ['b1', 'b2', 'bs', 'bk2', 'N_mp', 'N_pp']
    if which is None:
        which = joint_names

    # fit once per kmax, collect all 6 biases + errors
    vals, errs = [], []
    for km in kmaxes:
        popt, perr, _ = fit_joint(snpk, kmax=km)
        vals.append(popt)
        errs.append(perr)
    vals = np.array(vals)   # shape (n_kmax, 6)
    errs = np.array(errs)

    # plot each requested parameter vs kmax
    for idx, name in enumerate(joint_names):
        if name not in which:
            continue
        fig, ax = plt.subplots()
        ax.errorbar(kmaxes, vals[:, idx], yerr=errs[:, idx],
                    fmt='o-', capsize=3, capthick=1, elinewidth=1,
                    markersize=4, alpha=0.7, label='joint fit')
        ax.set_xlabel(r'$k_{\max}$ [h/Mpc]')
        ax.set_ylabel('Bias value')
        ax.set_title(f'{name} against k (joint)')
        ax.legend()

## Plots on Demand

In [100]:
feedback = ['L1_m9', 'fgas+2sigma', 'fgas-2sigma', 'fgas-4sigma', 'fgas-8sigma', 'Jet', 'Jet_fgas-4sigma', 'Mstar-1sigma', 'Mstar-1sigma_fgas-4sigma', 'NoCooling']
cosmology = ['Planck', 'PlanckNu0p24Fix', 'PlanckNu0p24Var', 'PlanckNu0p48Fix', 'LS8', 'LS8_fgas-8sigma']
spectra = ['Matter-Pressure', 'Pressure-Pressure']
biases = ['b1','b2','bs','bk2','Nmp', 'Npp']

In [126]:
def redshift_to_snapshot(target_z, zs_all, tol=0.01):
    diffs = np.abs(zs_all - target_z)
    idx = int(np.argmin(diffs))
    actual_z = zs_all[idx]
    exact = diffs[idx] <= tol
    return idx, actual_z, exact

In [127]:
def custom_fit_pk(snpk, tk, pk_data, pk_error, kmax=0.25, kmin=0.01, verbose=True, plot=True, nl = False):
    # Fit a power spectrum with the EFT templates using curve_fit
    a = 1/(1+snpk.z)

    # Fit a power spectrum with the EFT templates
    k = pk_data.kavg
    pk = pk_data.pk
    epk = pk_error  # Gaussian error bars

    # Select the data to fit
    goodk = (k >= kmin) & (k <= kmax)
    kfit = k[goodk]
    pkfit = pk[goodk]
    epkfit = epk[goodk]

    tk_b1_b1 = tk['b1:b1'](kfit, a)
    tk_b1_b2 = tk['b1:b2'](kfit, a)
    tk_b1_bs = tk['b1:bs'](kfit, a)
    tk_b1_bk2 = tk['b1:bk2'](kfit, a)
    tk_b2_b2 = tk['b2:b2'](kfit, a)
    tk_b2_bs = tk['b2:bs'](kfit, a)
    tk_bs_bs = tk['bs:bs'](kfit, a)

    # Define the model function to fit
    def model(k, b1, b2, bs, bk2, N):
        return (b1 * tk_b1_b1 +
                0.5*b2 * tk_b1_b2 +
                0.5*bs * tk_b1_bs +
                0.5*bk2 * tk_b1_bk2 + N)

    def nl_model(k, b1, b2, bs, bk2, N):
        return (b1**2 * tk_b1_b1 +
                b2*b1 * tk_b1_b2 +
                b1*bs * tk_b1_bs +
                0.25 * b2**2 * tk_b2_b2 + 
                0.5 * b2*bs * tk_b2_bs +
                0.25 * bs**2 * tk_bs_bs +
                b1*bk2 * tk_b1_bk2 +
                N)

    # Fit the model to the data using least squares
    if nl:
        popt, pcov = curve_fit(nl_model, kfit, pkfit, sigma=epkfit, maxfev=1000000) #Struggles to converge at default (maxfev = 1200)
        chi2 = np.sum(((pkfit - nl_model(kfit, *popt)) / epkfit) ** 2)
        ndof = len(pkfit) - len(popt)
        if verbose:
            print(f'Fit results: b1={popt[0]:.3e}, b2={popt[1]:.3e}, bs={popt[2]:.3e}, bk2={popt[3]:.3e}, N={popt[4]:.3e}, chi2/ndof={chi2/ndof:.2f}')
            print(popt)
        if plot:
            b1, b2, bs, bk2, N = popt
            amps = np.array([
                b1**2,          
                2*b1*b2,          
                2*b1*bs,          
                b2**2,    
                b2*bs,     
                bs**2,     
                2*b1*bk2,         
                N,              
            ])
            Tmat = np.array([tk_b1_b1, 0.5*tk_b1_b2, 0.5*tk_b1_bs,
                             0.25*tk_b2_b2, 0.5*tk_b2_bs, 0.25*tk_bs_bs,
                             0.5*tk_b1_bk2, np.ones_like(tk_b1_b1)]).T
            plot_fit(kfit, pkfit, epkfit, amps, Tmat, nl = True)
        return popt, pcov, chi2/ndof
    else:
        popt, pcov = curve_fit(model, kfit, pkfit, sigma=epkfit)
        chi2 = np.sum(((pkfit - model(kfit, *popt)) / epkfit) ** 2)
        ndof = len(pkfit) - len(popt)
        if verbose:
            print(f'Fit results: b1={popt[0]:.3e}, b2={popt[1]:.3e}, bs={popt[2]:.3e}, bk2={popt[3]:.3e}, N={popt[4]:.3e}, chi2/ndof={chi2/ndof:.2f}')
        if plot:
            Tmat = np.array([tk_b1_b1, 0.5*tk_b1_b2, 0.5*tk_b1_bs,
                             0.5*tk_b1_bk2, np.ones_like(tk_b1_b1)]).T
            plot_fit(kfit, pkfit, epkfit, popt, Tmat)
        return popt, pcov, chi2/ndof
    

def custom_fit_pk_analytic(snpk, tk, pk_data, pk_error, kmax=0.25, kmin=0.01, verbose=True, plot=True):
    # Fit a power spectrum with the EFT templates using the analytical least-squares solution
    a = 1/(1+snpk.z)

    # Fit a power spectrum with the EFT templates
    k = pk_data.kavg
    pk = pk_data.pk
    epk = pk_error  # Gaussian error bars

    # Select the data to fit
    goodk = (k >= kmin) & (k <= kmax)
    kfit = k[goodk]
    pkfit = pk[goodk]
    epkfit = epk[goodk]

    tk_b1_b1 = tk['b1:b1'](kfit, a)
    tk_b1_b2 = tk['b1:b2'](kfit, a)
    tk_b1_bs = tk['b1:bs'](kfit, a)
    tk_b1_bk2 = tk['b1:bk2'](kfit, a)


    
    tk_N = np.ones_like(tk_b1_b1)
    Tmat = np.array([tk_b1_b1, 0.5*tk_b1_b2, 0.5*tk_b1_bs, 0.5*tk_b1_bk2, tk_N]).T
    Cov = np.diag(epkfit**2)
    iCov = np.linalg.inv(Cov)
    TiCT = np.dot(Tmat.T, np.dot(iCov, Tmat))
    TiCp = np.dot(Tmat.T, np.dot(iCov, pkfit))
    pcov = np.linalg.inv(TiCT)
    popt = np.dot(pcov, TiCp)
    chi2 = np.sum(((pkfit - np.dot(Tmat, popt)) / epkfit) ** 2)
    ndof = len(pkfit) - len(popt)
    if verbose:
        print(f'Fit results: b1={popt[0]:.3e}, b2={popt[1]:.3e}, bs={popt[2]:.3e}, bk2={popt[3]:.3}, N={popt[4]:.3e}, chi2/ndof={chi2/ndof:.2f}')
    if plot:
        plot_fit(kfit, pkfit, epkfit, popt, Tmat)
    return popt, pcov, chi2/ndof

def custom_fit_joint(snpk, tk, kmax=0.25, kmin=0.01, verbose=True):
    
    a = 1/(1+snpk.z)
    k = snpk.pks['matter_pressure'].kavg
    goodk = (k >= kmin) & (k <= kmax)
    kfit = k[goodk]

    pk_mp  = snpk.pks['matter_pressure'].pk[goodk]
    epk_mp = snpk.epks['matter_pressure'][goodk]
    pk_pp  = snpk.pks['pressure_pressure'].pk[goodk]
    epk_pp = snpk.epks['pressure_pressure'][goodk]

    # evaluate the templates once, at kfit and a
    tk_b1_b1  = tk['b1:b1'](kfit, a)
    tk_b1_b2  = tk['b1:b2'](kfit, a)
    tk_b1_bs  = tk['b1:bs'](kfit, a)
    tk_b1_bk2 = tk['b1:bk2'](kfit, a)
    tk_b2_b2  = tk['b2:b2'](kfit, a)
    tk_b2_bs  = tk['b2:bs'](kfit, a)
    tk_bs_bs  = tk['bs:bs'](kfit, a)

    def joint_residual(params):
        b1, b2, bs, bk2, N_mp, N_pp = params

        mp = (b1 * tk_b1_b1 +
              0.5*b2 * tk_b1_b2 +
              0.5*bs * tk_b1_bs +
              0.5*bk2 * tk_b1_bk2 +
              N_mp)

        pp = (b1**2 * tk_b1_b1 +
              b1*b2 * tk_b1_b2 +
              b1*bs * tk_b1_bs +
              0.25*b2**2 * tk_b2_b2 +
              0.5*b2*bs * tk_b2_bs +
              0.25*bs**2 * tk_bs_bs +
              b1*bk2 * tk_b1_bk2 +
              N_pp)

        res_mp = (pk_mp - mp) / epk_mp
        res_pp = (pk_pp - pp) / epk_pp
        return np.concatenate([res_mp, res_pp])

    p0 = [1e-5, 0, 0, 0, 0, 0]
    result = least_squares(joint_residual, p0, method='trf')

    popt = result.x
    chi2 = np.sum(result.fun**2)                    
    ndof = len(result.fun) - len(popt)             
    cov  = np.linalg.inv(result.jac.T @ result.jac)
    perr = np.sqrt(np.diag(cov))

    if verbose:
        names = ['b1', 'b2', 'bs', 'bk2', 'N_mp', 'N_pp']
        for n, v, e in zip(names, popt, perr):
            print(f'{n} = {v:.3e} ± {e:.3e}')
        print(f'chi2/ndof = {chi2/ndof:.2f}')

    return popt, perr, chi2/ndof

def plot_joint_bias(snpk, tk, kmaxes, bias):
    joint_names = ['b1', 'b2', 'bs', 'bk2', 'N_mp', 'N_pp']
    idx = joint_names.index(bias)   # b1..bk2 map directly; 'N' would need care

    vals, errs = [], []
    for km in kmaxes:
        try:
            popt, perr, _ = custom_fit_joint(snpk, tk, kmax=km, verbose=False)
            vals.append(popt[idx]); errs.append(perr[idx])
        except (RuntimeError, np.linalg.LinAlgError):
            vals.append(np.nan); errs.append(np.nan)

    fig, ax = plt.subplots()
    ax.errorbar(kmaxes, vals, yerr=errs, fmt='o-', capsize=3, ms=4, alpha=0.7)
    ax.set_xlabel(r'$k_{\max}$ [h/Mpc]')
    ax.set_ylabel(f'{bias} value')
    ax.set_title(f'{snpk.sim} — {bias} (joint) — z={snpk.z:.2f}')

In [128]:
from itertools import product

In [129]:
def build_templates(cosmo):
    ept = ccl.nl_pt.EulerianPTCalculator(with_NC=True, sub_lowk=False, b1_pk_kind='pt', bk2_pk_kind='linear')
    ept.update_ingredients(cosmo)
    pairs = ['b1:b1','b1:b2','b1:bs','b1:bk2','b2:b2','b2:bs','bs:bs']
    return {p: ept.get_pk2d_template(p) for p in pairs}

fiducial = ccl.Cosmology(h=0.681, Omega_b=0.0486, Omega_c=0.306-0.0486,
                         n_s=0.967, sigma8=0.807)
cosmologies = {
    'fiducial': fiducial,
    'Planck': ccl.Cosmology(h=0.673, Omega_b=0.0494, Omega_c=0.316-0.0494, n_s=0.966, sigma8=0.812),         
    'Nu0p24Fix': ccl.Cosmology(h=0.673, Omega_b=0.0494, Omega_c=0.316-0.0494, n_s=0.966, sigma8=0.769),
    'Nu0p24Var': ccl.Cosmology(h=0.662, Omega_b=0.0510, Omega_c=	0.328-0.0510, n_s=0.968, sigma8=0.772),
    'Nu0p48Fix': ccl.Cosmology(h=0.673, Omega_b=0.0494, Omega_c=0.316-0.0494, n_s=0.966, sigma8=0.709),
    'LS8': ccl.Cosmology(h=0.682, Omega_b=0.0473, Omega_c=0.305-0.0473, n_s=0.965, sigma8=0.760)
}

templates_by_cosmo = {name: build_templates(c) for name, c in cosmologies.items()}

sim_cosmology = {
    'L1_m9':'fiducial', 'fgas+2sigma':'fiducial', 'fgas-2sigma':'fiducial',
    'fgas-4sigma':'fiducial', 'fgas-8sigma':'fiducial', 'Jet':'fiducial',
    'Jet_fgas-4sigma':'fiducial', 'Mstar-1sigma':'fiducial',
    'Mstar-1sigma_fgas-4sigma':'fiducial', 'NoCooling':'fiducial',
    'Planck':'Planck', 'PlanckNu0p24Fix':'Nu0p24Fix', 'PlanckNu0p24Var':'Nu0p24Var',
    'PlanckNu0p48Fix':'Nu0p48Fix', 'LS8':'LS8', 'LS8_fgas-8sigma':'LS8',
}

In [132]:
def run_plots(selection, kmaxes, templates_by_cosmo, sim_cosmology):
    
    zs_all = np.loadtxt("FLAMINGO/zs_list.txt")

    sims = selection['feedback'] + selection['cosmology']

    whats = selection['spectra'] + selection['biases']

    SPECTRA = {'Matter-Pressure': ('matter_pressure', False),
               'Pressure-Pressure': ('pressure_pressure', True)}
    BIAS_NAMES = ['b1', 'b2', 'bs', 'bk2', 'N']

    target_zs = [float(i) for i in selection['Redshifts'].split(',') if i.strip()]
    snapshots = []
    for z in target_zs:
        idx, actual_z, exact = redshift_to_snapshot(z, zs_all)
        if not exact:
            print(f'z = {z} not found; using nearest: snapshot {idx} '
                  f'(z = {actual_z:.3f}, off by {abs(actual_z - z):.3f})')
        snapshots.append(idx)

    for sim, what, sn in product(sims, whats, snapshots):
        tk = templates_by_cosmo[sim_cosmology[sim]]
        snpk = SnapPk(sim, sn, kmax=10)

        if what in SPECTRA:
            spec_key, nl = SPECTRA[what]
            custom_fit_pk(snpk, tk, snpk.pks[spec_key], snpk.epks[spec_key],
                   kmax=kmaxes[-1], verbose=False, plot=True, nl=nl)

        elif what in BIAS_NAMES:
            plot_joint_bias(snpk, tk, kmaxes, what)

        plt.show()

In [99]:
import ipywidgets as widgets
from IPython.display import display

def selectinator(feedback, cosmology, spectra, biases):
    def make_checkboxes(options, title):
        boxes = [widgets.Checkbox(value=False, description=str(o)) for o in options]
        return widgets.VBox([widgets.Label(title)] + boxes), boxes
    
    feed_box,  feed_cb  = make_checkboxes(feedback, 'Feedback')
    cosm_box,  cosm_cb  = make_checkboxes(cosmology, 'Cosmology')
    spec_box, spec_cb = make_checkboxes(spectra,     'Spectra')
    bias_box, bias_cb = make_checkboxes(biases,      'Biases')

    def make_input():
        text = widgets.Text(value='', placeholder='Type something', description='Redshifts', disabled=False)
        return text

    rs_input = make_input()
    
    button = widgets.Button(description='Confirm', button_style='success')
    out = widgets.Output()
    selection = {}

    def on_click(_):
        selection['feedback'] = [o for o, cb in zip(feedback, feed_cb)  if cb.value]
        selection['cosmology'] = [o for o, cb in zip(cosmology, cosm_cb)  if cb.value]
        selection['spectra']     = [o for o, cb in zip(spectra,     spec_cb) if cb.value]
        selection['biases']      = [o for o, cb in zip(biases,      bias_cb) if cb.value]
        selection['Redshifts'] = rs_input.value
        with out:
            out.clear_output()
            run_plots(selection, kmaxes, templates_by_cosmo, sim_cosmology)
    
    button.on_click(on_click)
    display(widgets.HBox([feed_box, cosm_box, spec_box, bias_box]), rs_input, button, out)
    return selection

## Isolating Plateaus (wip)

In [54]:
#We now want to isolate the plateaus where the biases are constant and agree. 

In [55]:
def compute_biases(snpk, kmaxes):
    names = ['b1', 'b2', 'bs', 'bk2', 'N']
    mp_val, mp_err, pp_val, pp_err = [], [], [], []

    for km in kmaxes:
        popt,    pcov,    _ = fit_pk(snpk, snpk.pks['matter_pressure'],
                                     snpk.epks['matter_pressure'],
                                     kmax=km, verbose=False, plot=False)
        popt_nl, pcov_nl, _ = fit_pk(snpk, snpk.pks['pressure_pressure'],
                                     snpk.epks['pressure_pressure'],
                                     kmax=km, verbose=False, plot=False, nl=True)
        mp_val.append(popt)
        mp_err.append(np.sqrt(np.diag(pcov)))
        pp_val.append(popt_nl)
        pp_err.append(np.sqrt(np.diag(pcov_nl)))

    return {
        'names':  names,
        'kmaxes': np.asarray(kmaxes),
        'mp_val': np.array(mp_val),   
        'mp_err': np.array(mp_err),
        'pp_val': np.array(pp_val),
        'pp_err': np.array(pp_err),
    }

In [56]:
#data = compute_biases(snpk, kmaxes)

In [57]:
def filter_agree(data, nsig=2):
    km = data['kmaxes']
    results = {}

    for j, name in enumerate(data['names']):
        b_mp, e_mp = data['mp_val'][:, j], data['mp_err'][:, j]
        b_pp, e_pp = data['pp_val'][:, j], data['pp_err'][:, j]

        #Check the two spectra agree within combined error
        agree = np.abs(b_mp - b_pp) <= nsig * np.sqrt(e_mp**2 + e_pp**2)

        mask = agree 
        best_len, best = 0, (None, None)
        i = 0
        while i < len(mask):
            if mask[i]:
                j0 = i
                while i < len(mask) and mask[i]:
                    i += 1
                if i - j0 > best_len:
                    best_len, best = i - j0, (j0, i - 1)
            else:
                i += 1

        results[name] = best
    return results

In [58]:
def filtered_coefficient_plotter(data, windows=None, which=None):
    names = data['names']
    km = data['kmaxes']
    if which is None:
        which = names

    for j, name in enumerate(names):
        if name not in which:
            continue

        b_mp, e_mp = data['mp_val'][:, j], data['mp_err'][:, j]
        b_pp, e_pp = data['pp_val'][:, j], data['pp_err'][:, j]

        fig, ax = plt.subplots()
        ax.errorbar(km, b_mp, yerr=e_mp, label='matter-pressure',
                    fmt='o-', capsize=3, capthick=1, elinewidth=1, markersize=4, alpha=0.7)
        ax.errorbar(km, b_pp, yerr=e_pp, label='pressure-pressure',
                    fmt='o-', capsize=3, capthick=1, elinewidth=1, markersize=4, alpha=0.7)

        # shade the agreement window if provided
        if windows is not None and windows[name][0] is not None:
            lo, hi = windows[name]
            ax.axvspan(km[lo], km[hi], alpha=0.15, color='green', label='agreement window')

        ax.set_xlabel(r'$k_{\max}$ [h/Mpc]')
        ax.set_ylabel('Bias value')
        ax.set_title(f'{name} against k')
        ax.legend()